In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bumba5341/advertisingcsv")

print("Path to dataset files:", path)

In [ ]:
%%bash
# Create the directory if it doesn't exist
mkdir -p ~/Downloads

# Download the dataset
curl -L -o ~/Downloads/q3-ka-ai-2026.zip \
  https://www.kaggle.com/api/v1/datasets/download/mohammad2012191/q3-ka-ai-2026

# Unzip the file to access the CSV
unzip -o ~/Downloads/q3-ka-ai-2026.zip -d ./data_folder

In [ ]:
import pandas as pd
# Task 1-2
# Path depends on where i unzipped it in the bash step
df = pd.read_csv('./data_folder/Q3_data.csv')

print(f"Dataset Shape: {df.shape}")
print(df.head())

In [ ]:
# Task 3: Display dataset information
df.info()

In [ ]:
# Task 4: Show statistical description
print(df.describe())

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Task 1: Handle missing values (Filling with median for numerical columns)
df = df.fillna(df.median(numeric_only=True))


In [ ]:
# Task 2: Check and remove duplicates
df = df.drop_duplicates()

In [ ]:
# Task 3: Encode categorical variables (Simple OHE or Label Encoding if needed)
# Note: CatBoost handles categories well, but for scaling, we'll ensure they are numeric
df = pd.get_dummies(df, drop_first=True)


In [ ]:
print(df.columns)

In [ ]:
# Task 4: Apply feature scaling
scaler = StandardScaler()

# Using the correct capitalized name from your output
target_col = 'Target'

features = df.drop(target_col, axis=1)
scaled_features = scaler.fit_transform(features)

# Create the scaled dataframe
df_scaled = pd.DataFrame(scaled_features, columns=features.columns)
print("Scaling successful!")

# Task 5: Check for target imbalance
# We use the original 'df' for the target column
imbalance_ratio = df[target_col].value_counts(normalize=True)
print(f"\nTarget Distribution:\n{imbalance_ratio}")
print("-" * 30)
print("Result: Imbalanced" if imbalance_ratio.min() < 0.4 else "Result: Balanced")

In [ ]:
# Task 1: Split the dataset
X = df_scaled
y = df['Target']  # Updated to capital 'T'

# Tasks 2: StratifiedKFold and CatBoost
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

print("Starting Cross-Validation...")

for fold, (train_index, test_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize CatBoost
    model = CatBoostClassifier(verbose=0, n_estimators=500, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate using F1 Score (better for credit risk than Accuracy)
    y_pred = model.predict(X_test)
    score = f1_score(y_test, y_pred)
    scores.append(score)
    print(f"Fold {fold} F1 Score: {score:.4f}")

print(f"\nFinal Averaged F1 Score: {np.mean(scores):.4f}")

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np


In [ ]:
# Task 1: Split the dataset
X = df_scaled
y = df['target']

In [ ]:
# Tasks 2: StratifiedKFold and CatBoost
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(verbose=0, n_estimators=500)
    model.fit(X_train, y_train)

    # Using F1 Score due to potential credit default imbalance
    y_pred = model.predict(X_test)
    scores.append(f1_score(y_test, y_pred))

print(f"Averaged F1 Score: {np.mean(scores):.4f}")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Task 1: Plot feature importance
feature_importance = model.get_feature_importance()
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 8))
plt.barh(X.columns[sorted_idx][-10:], feature_importance[sorted_idx][-10:])
plt.xlabel("CatBoost Feature Importance")
plt.title("Top 10 Features")
plt.show()

In [ ]:
# Task 2: Identify the Golden Feature
golden_feature = X.columns[sorted_idx][-1]
print(f"The Golden Feature is: {golden_feature}")

In [ ]:
# Task Bonus: Retrain using ONLY the golden feature
X_golden = X[[golden_feature]]
golden_scores = []

for train_index, test_index in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    g_model = CatBoostClassifier(verbose=0, n_estimators=500)
    g_model.fit(X_train, y_train)

    y_pred = g_model.predict(X_test)
    golden_scores.append(f1_score(y_test, y_pred))

print(f"Full Model F1: {np.mean(scores):.4f}")
print(f"Golden Feature Only F1: {np.mean(golden_scores):.4f}")